In [1]:
import os
os.chdir("..")


In [2]:
import pandas as pd
from imap_tools import AND, UidRange
from tqdm import trange, tqdm
from collections import defaultdict
import socket
import ssl
import os
from src.imap.IMAPClientOperations import IMAPClientOperations as IMAPClient

In [3]:
EMAIL = "info@geekprom.ru"
BATCH_SIZE = 20
conversations = defaultdict(lambda: {'original': None, 'reply': None})

In [4]:
client = IMAPClient().client

[2025-08-20 19:54:49.349958][IMAP][INFO] Connected to imap.yandex.ru:993
[2025-08-20 19:54:49.657289][IMAP][INFO] Login successfully


In [9]:
import pandas as pd
from imap_tools import MailBox, AND
from tqdm import tqdm
import csv
import os

def imap_to_csv(folder: str = 'INBOX',
                csv_filename: str = 'emails.csv',
                batch_size: int = 100):
    try:
        total_emails = int(client.folder.set(folder)[1][0])
        print(f"Найдено писем: {total_emails}")
        
        fieldnames = [
            'uid', 'subject', 'from', 'in_reply_to', 'to', 'date', 'text', 'html',
            'flags', 'size', 'attachments_count', 'headers'
        ]
        
        # Открываем CSV файл для записи
        with open(csv_filename, 'a', newline='\n', encoding='utf-8') as csvfile:
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            
            # Создаем прогресс-бар
            with tqdm(total=total_emails, desc="Обработка писем", unit="письмо") as pbar:
                
                batch_count = 0
                emails_processed = 0
                
                # Обрабатываем письма батчами
                for msg in client.fetch(AND(all=True), bulk=5, mark_seen=False):
                    email_data = {
                        'uid': msg.uid,
                        'subject': msg.subject or '',
                        'from': msg.from_ or '',
                        'in_reply_to': msg.reply_to or '',
                        'to': ', '.join(msg.to) if msg.to else '',
                        'date': msg.date.strftime('%Y-%m-%d %H:%M:%S') if msg.date else '',
                        'text': msg.text or '',
                        'html': msg.html or '',
                        'flags': ', '.join(msg.flags) if msg.flags else '',
                        'size': msg.size,
                        'attachments_count': len(msg.attachments),
                        'headers': str(msg.headers) if msg.headers else ''
                    }
                    
                    # Записываем данные
                    writer.writerow(email_data)
                    
                    emails_processed += 1
                    batch_count += 1
                    pbar.update(1)
                    
                    # Обновляем прогресс каждые batch_size писем
                    if batch_count >= batch_size:
                        pbar.set_postfix({
                            'обработано': f'{emails_processed}/{total_emails}',
                            'батч': batch_count
                        })
                        batch_count = 0
        
        print(f"Данные сохранены в файл: {csv_filename}")
        
    except Exception as e:
        print(f"Ошибка при обработке писем: {e}")

In [10]:

print("="*50)
print(f"Старт обработки почты {EMAIL}")
print("="*50)

imap_to_csv(folder='ОТВЕЧЕНО', csv_filename='./data_collecting/answered.csv')
imap_to_csv(folder='Sent', csv_filename='./data_collecting/sent.csv')



Старт обработки почты info@geekprom.ru
Найдено писем: 31282


Обработка писем:   0%|          | 55/31282 [00:23<3:44:36,  2.32письмо/s]


KeyboardInterrupt: 

,uid,subject,from,in_reply_to,to,date,text,html,flags,size,attachments_count,headers
0,1,Fwd: биннофарм 1016,info@sbr-promimport.ru,NaN,info@geekprom.ru,2023-10-25 11:37:25,"Игорь, здравствуйте.\r\nПо конкурентной цене м...",<div> </div><div> </div><div>-------- Пересыла...,"\Seen, encrypted, system_hamon",784913,3,{'received': ('from postback10b.mail.yandex.ne...
1,2,запрос срочный,gkazia@yandex.ru,NaN,info@geekprom.ru,2023-10-25 17:55:30,NaN,<div> </div><div><div>Добрый день!</div><div>П...,"\Seen, \Answered, encrypted, system_hamon",37778,1,{'received': ('from postback16a.mail.yandex.ne...
2,3,RE: Датчики ПРОМПРИБОР-Р,eafanasev@binnopharmgroup.ru,NaN,info@geekprom.ru,2023-10-26 04:56:33,"Алексей Михайлович, спасибо.\r\nВот минимальна...","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...","\Seen, \Answered, encrypted, system_hamon",20314,0,{'received': ('from postback19c.mail.yandex.ne...
3,4,RE: Датчики ПРОМПРИБОР-Р,eafanasev@binnopharmgroup.ru,NaN,info@geekprom.ru,2023-10-26 09:21:24,"Алексей, спасибо.\r\nВот минимальная стоимость...","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...","\Seen, encrypted, system_hamon",33543,0,{'received': ('from postback6a.mail.yandex.net...
4,5,Re: Поставка оборудования СПО Аналитприбор г.С...,m1.npk.ett@gmail.com,NaN,"lfc05@mail.ru, info@geekprom.ru",2023-10-25 09:30:53,"Добрый день!\r\n\r\nПрошу прислать договор, в ...","<div dir=""ltr"">Добрый день!<br><br>Прошу присл...","\Seen, \Answered, encrypted, system_hamon",78488,1,{'received': ('from postback9c.mail.yandex.net...
5,6,газосигнализатор ГСА-3,6254025@kuz.myjino.ru,NaN,info@geekprom.ru,2023-10-24 14:32:54,NaN,<p>Прошу Вас предоставить Коммерческое предлож...,"\Seen, \Answered, encrypted, system_hamon",86411,2,{'received': ('from postback22b.mail.yandex.ne...
6,7,Re: запрос срочный,gkazia@yandex.ru,NaN,info@geekprom.ru,2023-10-26 14:40:57,NaN,<div>там первая позиция нужен не сам прибор а ...,"\Seen, \Answered, encrypted, system_hamon",9805,0,{'received': ('from postback5b.mail.yandex.net...
7,8,Re: Поставка оборудования СПО Аналитприбор г.С...,m1.npk.ett@gmail.com,NaN,info@geekprom.ru,2023-10-27 09:12:28,Добрый день!\r\n\r\nСлужба безопасности не про...,"<div dir=""ltr"">Добрый день!<br><br><div>Служба...","\Seen, \Answered, encrypted, system_hamon",26199,0,{'received': ('from postback19c.mail.yandex.ne...
8,9,Сигнализатор загазованности стационарный шлейф...,sale@teka-group.ru,NaN,info@geekprom.ru,2023-10-30 13:47:25,NaN,"<div><div style=""background-color:rgb( 255 , 2...","\Seen, \Answered, encrypted",29843,1,{'received': ('from postback30a.mail.yandex.ne...
9,10,RE: СИГНАЛ-03,anb@serviceazs.ru,NaN,info@geekprom.ru,2023-11-01 07:04:02,"Спасибо!!!\r\n\r\n \r\n\r\nFrom: ООО ""ГИКПРОМ""...","<html xmlns:v=""urn:schemas-microsoft-com:vml"" ...","\Seen, encrypted, system_hamon",27424,0,{'received': ('from postback30b.mail.yandex.ne...
